In this notebook we process the data from the MD-Simulations

In [1]:
import pandas as pd

# We drop all duplicate molecules and keep the best (=lowest) score
def clean_dataset(df):
    n_duplicates = df["smiles"].duplicated().sum()
    print(f"Dataset contains {n_duplicates} duplicates.")
    
    assert not df["target"].isna().any(), "Dataset contains molecules without a score."
    
    clean_df = df.sort_values(by="target").drop_duplicates(["smiles"], keep="first")
    # Avoid any information leakage by having the data ordered
    clean_df = clean_df.sample(frac=1.0, random_state=0).reset_index(drop=True)
    return clean_df

We start by cleaning the Enamine datasets that have been used in the literature to evaluate active learning.

In [2]:
for ds in ["unprocessed_Enamine10k_scores.csv", "unprocessed_Enamine50k_scores.csv"]:
    df = pd.read_csv(ds)
    clean_df = clean_dataset(df)
    
    clean_df.to_csv(ds.removeprefix("unprocessed_"), index=False)

Dataset contains 3 duplicates.
Dataset contains 7 duplicates.


The results from docking and MMGBSA/MMPBSA have been prepared in the `results.csv` file. We now extract the necessary columns for bayesian optimization.

In [ ]:
results_df = pd.read_csv("results.csv")
results_df = results_df.drop(columns=["orig_smiles"])
results_df = results_df.rename(columns={"prot_smiles": "smiles"})
results_df = results_df.sample(frac=1, random_state=0)  # Shuffle to avoid any potential bias by being sorted by mmgbsa_score
results_df.head()

,name,smiles,mmgbsa_score,mmgbsa_score_sem,mmpbsa_score,mmpbsa_score_sem,vina_score
39136,ZINCqw00000qSaMx,CC(C)[C@@H](NC(=O)c1[nH]c2c(Br)cccc2c1CCCO)C(=...,-28.061157,0.452952,-21.048346,0.279807,-7.886
2819,ZINCqH00000vdU8F,CC1(C)CCC(=O)[C@H](OC(=O)CC[C@@H]2CCN(C(=O)OC(...,-40.371413,0.300698,-29.865634,0.348530,-8.057
5007,ZINCnH00000iybgH,CO[C@H]1C[C@@H](OC(=O)CCc2c(C)[nH]c3ccccc23)C1...,-38.625181,0.403245,-30.356945,0.237431,-8.658
4849,ZINCpM00000SL3Ft,Cc1[nH]cc(CCC(=O)Oc2cccc3scc(Br)c23)c1S(C)(=O)=O,-38.727840,0.524762,-31.158861,0.377273,-8.878
48470,ZINCnp00000F0gVz,COC(=O)Cc1c(C(=O)OC/C(C)=C/CO)[nH]c2ccccc12,-23.042376,0.868263,-21.084870,0.707793,-7.509


In [4]:
results_df[["name", "smiles", "mmgbsa_score"]].rename(columns={"mmgbsa_score":"target"}).to_csv("MCL1-mmgbsa.csv")
results_df[["name", "smiles", "mmpbsa_score"]].rename(columns={"mmpbsa_score":"target"}).to_csv("MCL1-mmpbsa.csv")
results_df[["name", "smiles", "vina_score"]].rename(columns={"vina_score":"target"}).to_csv("MCL1-vina.csv")